# 01 - Federated data partition (Paper 4, Fase 1)

**Tujuan:** tiap *klien* menerapkan kamus **SFM (public prior dari Paper 1)** ke
datanya SENDIRI -> 9 fitur SFM lokal. Client-CIC & Client-UNSW TIDAK bertukar data;
yang disepakati hanya *skema* (9 fitur + definisi). Lihat `documentation_federated.md`
Bagian A.1 (SFM = public prior, bukan hasil belajar data gabungan).

**Output:** `fed_out/dataset_stats_fed.csv` (statistik per-klien, ANGKA NYATA) +
partisi SFM per-klien (`fed_out/*.npz`) + unggah ke S3 `unsw-far/federated/data/{cic,unsw}/`.

**PENTING (skala CIC):** `cleaned_100.pkl` menyimpan fitur yang SUDAH di-StandardScaler
(pelajaran Paper 1). Notebook ini **mengembalikan skala asli** CIC via
`X_orig = X_scaled * scale_ + mean_` SEBELUM pemetaan SFM, agar sepadan dgn UNSW (raw).

> Jalankan di SageMaker (butuh CIC `cleaned_100.pkl` + UNSW CSV). JANGAN mengarang angka.

## 0. Setup & kamus SFM (public prior, warisi Paper 1)

In [ ]:
import importlib, sys, subprocess
for m in ('pandas','numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, pickle, numpy as np, pandas as pd
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); REGION='ap-southeast-1'
SEED=42

# --- Kamus SFM (public prior; identik Paper 1). (CIC_col, UNSW_col) per fitur ---
SFM = {
  'duration' : ('Flow Duration',   'dur'),
  'fwd_pkts' : ('Tot Fwd Pkts',    'spkts'),
  'bwd_pkts' : ('Tot Bwd Pkts',    'dpkts'),
  'fwd_bytes': ('TotLen Fwd Pkts', 'sbytes'),
  'bwd_bytes': ('TotLen Bwd Pkts', 'dbytes'),
  'fwd_mean' : ('Fwd Pkt Len Mean','smean'),
  'bwd_mean' : ('Bwd Pkt Len Mean','dmean'),
  'src_load' : ('Flow Byts/s',     'sload'),
  'dst_load' : ('Bwd Pkts/s',      'dload'),
}
SFM_FEATS=list(SFM.keys())
print('9 fitur SFM:', SFM_FEATS)

## 0b. Lokasi data + PEMERIKSA PRASYARAT (hard-stop bila data tak ada)

nb01 mencoba beberapa lokasi lokal untuk tiap berkas; bila CIC tak ada secara lokal,
mencoba unduh dari S3 (`unsw-far/CICDDoS2018/...` atau env `CIC_S3_KEY`). Bila kedua
dataset tetap tak lengkap -> **berhenti dgn pesan jelas** (jangan lanjut mengarang).

In [ ]:
def first_existing(paths):
    for p in paths:
        if p and os.path.exists(p): return p
    return None
UNSW_TRAIN=first_existing(['../../unswnb-15/data/UNSW_NB15_training-set.csv',
                           '../unswnb-15/data/UNSW_NB15_training-set.csv',
                           os.environ.get('UNSW_TRAIN')])
UNSW_TEST =first_existing(['../../unswnb-15/data/UNSW_NB15_testing-set.csv',
                           '../unswnb-15/data/UNSW_NB15_testing-set.csv',
                           os.environ.get('UNSW_TEST')])
CIC_PKL=first_existing(['../../CICDDoS2018/data/cleaned_100.pkl',
                        '../CICDDoS2018/data/cleaned_100.pkl',
                        'cleaned_100.pkl', os.environ.get('CIC_PKL')])
# Fallback: unduh CIC dari S3 bila tak ada lokal
if CIC_PKL is None:
    key=os.environ.get('CIC_S3_KEY','unsw-far/CICDDoS2018/cleaned_100.pkl')
    try:
        import boto3; s3=boto3.client('s3',region_name=REGION)
        dst=os.path.join(OUTDIR,'cleaned_100.pkl')
        print('CIC tak ada lokal -> unduh s3://%s/%s'%(S3_BUCKET,key))
        s3.download_file(S3_BUCKET,key,dst); CIC_PKL=dst
    except Exception as e:
        print('(unduh CIC dari S3 gagal:',e,')')
print('UNSW_TRAIN:', UNSW_TRAIN)
print('UNSW_TEST :', UNSW_TEST)
print('CIC_PKL   :', CIC_PKL)
missing=[n for n,v in [('UNSW_TRAIN',UNSW_TRAIN),('UNSW_TEST',UNSW_TEST),('CIC_PKL',CIC_PKL)] if v is None]
assert not missing, ('PRASYARAT GAGAL - berkas tak ditemukan: %s. '
                     'Set path via env (UNSW_TRAIN/UNSW_TEST/CIC_PKL/CIC_S3_KEY) '
                     'atau jalankan di SageMaker tempat data tersedia.'%missing)
print('OK semua prasyarat data tersedia.')

## 1. Client-UNSW: terapkan SFM ke data UNSW SENDIRI

UNSW train/test = dua berkas rilis resmi (disjoint). Tiap klien olah lokal.

In [ ]:
def unsw_to_sfm(df):
    cols={k:v[1] for k,v in SFM.items()}
    miss=[cols[k] for k in SFM_FEATS if cols[k] not in df.columns]
    if miss: print('WARN kolom UNSW hilang:', miss)
    out=pd.DataFrame({k: pd.to_numeric(df.get(cols[k]), errors='coerce') for k in SFM_FEATS})
    out=out.replace([np.inf,-np.inf], np.nan).fillna(out.median(numeric_only=True)).fillna(0.0)
    y=df['label'].astype(int).values
    return out.astype(float), y
utr=pd.read_csv(UNSW_TRAIN); ute=pd.read_csv(UNSW_TEST)
Xu_tr,yu_tr=unsw_to_sfm(utr); Xu_te,yu_te=unsw_to_sfm(ute)
print('UNSW train', Xu_tr.shape, 'pos', round(float(yu_tr.mean()),3),
      '| test', Xu_te.shape, 'pos', round(float(yu_te.mean()),3))

## 2. Client-CIC: muat (un-scale) + terapkan SFM ke data CIC SENDIRI

`cleaned_100.pkl` berisi objek sklearn -> pakai unpickler aman bila perlu, lalu
**kembalikan skala asli** sebelum SFM. Label CIC dideteksi fleksibel
(`y` + `label_mapping`/`label_names`), diubah ke biner attack=1/benign=0.
Split stratified 70/30 (seed tetap).

In [ ]:
import io
class _SafeUnpickler(pickle.Unpickler):
    # fallback bila objek sklearn tak dapat diimpor: ganti dgn stub, cukup utk ambil X/y/scaler meta
    def find_class(self, module, name):
        try: return super().find_class(module, name)
        except Exception: return type(name, (object,), {})
def _load_pkl(path):
    with open(path,'rb') as f:
        try: return pickle.load(f)
        except ModuleNotFoundError as e:
            print('[warn] load penuh gagal (%s); pakai SafeUnpickler'%e)
            f.seek(0); return _SafeUnpickler(f).load()
def cic_to_sfm(df):
    cols={k:v[0] for k,v in SFM.items()}
    miss=[cols[k] for k in SFM_FEATS if cols[k] not in df.columns]
    if miss: print('WARN kolom CIC hilang:', miss)
    out=pd.DataFrame({k: pd.to_numeric(df.get(cols[k]), errors='coerce') for k in SFM_FEATS})
    out=out.replace([np.inf,-np.inf],np.nan).fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float)
d=_load_pkl(CIC_PKL)
cic_feats=list(d['feature_names'])
X=np.asarray(d['X'],dtype=float)
scaler=d.get('scaler',None)
if scaler is not None and hasattr(scaler,'scale_') and hasattr(scaler,'mean_'):
    X=X*np.asarray(scaler.scale_,float)+np.asarray(scaler.mean_,float)
    print('CIC: un-scaled ke skala asli (scaler.mean_/scale_)')
else:
    print('CIC: scaler tak tersedia -> X apa adanya (periksa interpretasi!)')
cic_df=pd.DataFrame(X,columns=cic_feats)
# label biner: attack=1, benign=0
y_raw=np.asarray(d['y'])
benign=None
for keyname in ('label_mapping','label_names','labels'):
    m=d.get(keyname)
    if isinstance(m,dict):
        for k,v in m.items():
            if str(k).lower().startswith('benign'): benign=v
    elif isinstance(m,(list,tuple)):
        for i,v in enumerate(m):
            if str(v).lower().startswith('benign'): benign=i
if benign is None: benign=0  # asumsi default; dicetak utk verifikasi
yc_all=(y_raw!=benign).astype(int)
print('CIC benign-code dipakai:', benign, '| pos-rate', round(float(yc_all.mean()),3))
from sklearn.model_selection import train_test_split
Xc_all=cic_to_sfm(cic_df)
Xc_tr,Xc_te,yc_tr,yc_te=train_test_split(Xc_all,yc_all,test_size=0.3,random_state=SEED,stratify=yc_all)
print('CIC train (SEBELUM subsample)', Xc_tr.shape, 'test', Xc_te.shape)

# === RUN-2 FIX: subsample CIC train ke ~100k (stratified) ===
# Akar masalah run-1: rasio CIC:UNSW = 14:1 -> FedAvg didominasi CIC -> UNSW kolaps.
# Subsample stratified menjaga pos-rate; rasio jadi ~1.2:1 dgn UNSW 82k.
TARGET_CIC = 100_000
if len(Xc_tr) > TARGET_CIC:
    from sklearn.utils import resample
    idx_sub = resample(np.arange(len(Xc_tr)), n_samples=TARGET_CIC, replace=False,
                       stratify=yc_tr, random_state=SEED)
    Xc_tr = Xc_tr.iloc[idx_sub].reset_index(drop=True)
    yc_tr = yc_tr[idx_sub]
    print('CIC train (SETELAH subsample stratified):', Xc_tr.shape,
          'pos-rate', round(float(yc_tr.mean()),3),
          '| rasio CIC:UNSW ~', round(len(Xc_tr)/len(Xu_tr),1), ':1')
else:
    print('CIC train sudah <= TARGET, tidak di-subsample')

## 3. Statistik per-klien (ANGKA NYATA) -> tabel dataset Paper 4

In [ ]:
rows=[]
def stat(name, X, y, scheme):
    n=len(y); atk=int(np.sum(np.asarray(y)==1)); ben=int(np.sum(np.asarray(y)==0))
    rows.append({'client':name,'total':n,'attack':atk,'benign':ben,
                 'pos_rate':round(atk/n,4) if n else 0.0,'scheme':scheme})
stat('Client-UNSW (train)',Xu_tr,yu_tr,'per-dataset (UNSW release train)')
stat('Client-UNSW (test)', Xu_te,yu_te,'held-out UNSW test')
stat('Client-CIC (train)', Xc_tr,yc_tr,'per-dataset (CIC 70/30 seed)')
stat('Client-CIC (test)',  Xc_te,yc_te,'held-out CIC test')
dfst=pd.DataFrame(rows); dfst.to_csv(os.path.join(OUTDIR,'dataset_stats_fed.csv'),index=False)
import IPython.display as ipd; ipd.display(dfst)
print('=== SEL 3 (statistik per-klien) SELESAI ===')

## 4. Simpan partisi SFM per-klien (lokal + S3)

Tiap partisi = data SATU klien saja (privasi: tak digabung). Juga tulis
`{cic,unsw}_train.npz` dgn nama yang diharapkan `aws/fed_node.py` (klien EC2).

In [ ]:
def save_part(prefix, X, y):
    if X is None: return
    p=os.path.join(OUTDIR, prefix+'.npz')
    Xv=X.values if hasattr(X,'values') else np.asarray(X,float)
    np.savez_compressed(p, X=np.asarray(Xv,float), y=np.asarray(y,int), feats=np.array(SFM_FEATS))
    print('saved', p, np.asarray(Xv).shape)
for nm,Xv,yv in [('unsw_train',Xu_tr,yu_tr),('unsw_test',Xu_te,yu_te),
                 ('cic_train',Xc_tr,yc_tr),('cic_test',Xc_te,yc_te)]:
    save_part(nm, Xv, yv)
# Unggah ke S3 (boto3; JANGAN aws s3 cp di instance - pelajaran Paper 2).
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in os.listdir(OUTDIR):
        if f.endswith('.npz') or f=='dataset_stats_fed.csv':
            sub='unsw' if f.startswith('unsw') else ('cic' if f.startswith('cic') else 'meta')
            s3.upload_file(os.path.join(OUTDIR,f),S3_BUCKET,f'unsw-far/federated/data/{sub}/{f}'); up+=1
    print('upload',up,'-> s3://%s/unsw-far/federated/data/'%S3_BUCKET)
except Exception as e: print('(upload S3 dilewati:',e,')')
print('=== SEL 4 (simpan partisi) SELESAI ===')

## 5. Catatan metodologi (untuk naskah)

- Kedua klien memakai **kamus SFM yang sama (public prior)** tetapi menerapkannya
  pada datanya MASING-MASING -> tak ada pertukaran data mentah (privasi FL terjaga).
- CIC dikembalikan ke **skala asli** sebelum SFM agar sepadan dgn UNSW (raw);
  normalisasi z-score dilakukan **per-klien lokal** di Fase 2 (opsi (a) Bagian A.1).
- Test set per-klien disimpan terpisah; **evaluasi global** memakai gabungan test
  (held-out) HANYA di notebook evaluasi (nb02-nb07), bukan data latih klien.
- `cic_train.npz`/`unsw_train.npz` dipakai juga oleh node EC2 (`aws/fed_node.py`).